# Proyecto Integrador de Aprendizaje Automático

Para el mercado de autos usados el precio de un vehículo dependen de múltiples factores. Con el uso de técnicas de Machine Learning vamos a construir modelos predictivos capaces de estimar el precio de un vehículo en función de sus características y saber si tendrá alta o baja demanda. 
Por ello, vamos a implementar modelos de regresión y clasificación supervisada, utilizando Ridge y Lasso, así como regresión logística. A través de un flujo completo con Pipeline y ajuste de hiperparámetros con GridSearchCV, se busca garantizar soluciones generalizables.

Esto va a ser de gran ayuda pues para los compradores y vendedores tomar decisiones rápidas y acertadas resultan importantes para evaluar de manera precisa el valor de un automóvil y su potencial. 
Por ello, este proyecto tiene un impacto práctico en tal pues ofrece predicciones útiles para la toma de decisiones estratégicas.


Para ello, iniciamos importando las librerias que necesitamos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

Leemos nuestro dataset sugerido

In [6]:
dataset = pd.read_csv('vehicles.csv', sep=',')

El tamaño de nuestro dataset:

In [4]:
print("Dimensiones:", dataset.shape)

Dimensiones: (426880, 26)


Revisamos las primeras 5 filas del dataset:

In [5]:
dataset.head()

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN


Las 5 últimas filas:

In [9]:
dataset.tail()

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
426875,7301591192,https://wyoming.craigslist.org/ctd/d/atlanta-2...,wyoming,https://wyoming.craigslist.org,23590,2019.0,nissan,maxima s sedan 4d,good,6 cylinders,...,NaN,sedan,NaN,https://images.craigslist.org/00o0o_iiraFnHg8q...,Carvana is the safer way to buy a car During t...,NaN,wy,33.786500,-84.445400,2021-04-04T03:21:31-0600
426876,7301591187,https://wyoming.craigslist.org/ctd/d/atlanta-2...,wyoming,https://wyoming.craigslist.org,30590,2020.0,volvo,s60 t5 momentum sedan 4d,good,NaN,...,NaN,sedan,red,https://images.craigslist.org/00x0x_15sbgnxCIS...,Carvana is the safer way to buy a car During t...,NaN,wy,33.786500,-84.445400,2021-04-04T03:21:29-0600
426877,7301591147,https://wyoming.craigslist.org/ctd/d/atlanta-2...,wyoming,https://wyoming.craigslist.org,34990,2020.0,cadillac,xt4 sport suv 4d,good,NaN,...,NaN,hatchback,white,https://images.craigslist.org/00L0L_farM7bxnxR...,Carvana is the safer way to buy a car During t...,NaN,wy,33.779214,-84.411811,2021-04-04T03:21:17-0600
426878,7301591140,https://wyoming.craigslist.org/ctd/d/atlanta-2...,wyoming,https://wyoming.craigslist.org,28990,2018.0,lexus,es 350 sedan 4d,good,6 cylinders,...,NaN,sedan,silver,https://images.craigslist.org/00z0z_bKnIVGLkDT...,Carvana is the safer way to buy a car During t...,NaN,wy,33.786500,-84.445400,2021-04-04T03:21:11-0600
426879,7301591129,https://wyoming.craigslist.org/ctd/d/atlanta-2...,wyoming,https://wyoming.craigslist.org,30590,2019.0,bmw,4 series 430i gran coupe,good,NaN,...,NaN,coupe,NaN,https://images.craigslist.org/00Y0Y_lEUocjyRxa...,Carvana is the safer way to buy a car During t...,NaN,wy,33.779214,-84.411811,2021-04-04T03:21:07-0600


Vemos que siguen en orden la información. Ahora vemos la naturaleza de los datos:

In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   url           426880 non-null  object 
 2   region        426880 non-null  object 
 3   region_url    426880 non-null  object 
 4   price         426880 non-null  int64  
 5   year          425675 non-null  float64
 6   manufacturer  409234 non-null  object 
 7   model         421603 non-null  object 
 8   condition     252776 non-null  object 
 9   cylinders     249202 non-null  object 
 10  fuel          423867 non-null  object 
 11  odometer      422480 non-null  float64
 12  title_status  418638 non-null  object 
 13  transmission  424324 non-null  object 
 14  VIN           265838 non-null  object 
 15  drive         296313 non-null  object 
 16  size          120519 non-null  object 
 17  type          334022 non-null  object 
 18  pain

Vemos que hay 7 variables numéricas y 19 variables categóricas. Aplicando esta función vemos que puede haber presencia de datos faltantes, por ello vamos a ver en qué variables se están presentando dichos faltantes.

In [7]:
dataset.describe()

,id,price,year,odometer,county,lat,long
count,4.268800e+05,4.268800e+05,425675.000000,4.224800e+05,0.0,420331.000000,420331.000000
mean,7.311487e+09,7.519903e+04,2011.235191,9.804333e+04,NaN,38.493940,-94.748599
std,4.473170e+06,1.218228e+07,9.452120,2.138815e+05,NaN,5.841533,18.365462
min,7.207408e+09,0.000000e+00,1900.000000,0.000000e+00,NaN,-84.122245,-159.827728
25%,7.308143e+09,5.900000e+03,2008.000000,3.770400e+04,NaN,34.601900,-111.939847
50%,7.312621e+09,1.395000e+04,2013.000000,8.554800e+04,NaN,39.150100,-88.432600
75%,7.315254e+09,2.648575e+04,2017.000000,1.335425e+05,NaN,42.398900,-80.832039
max,7.317101e+09,3.736929e+09,2022.000000,1.000000e+07,NaN,82.390818,173.885502


In [10]:
dataset.isnull().sum()

id                   0
url                  0
region               0
region_url           0
price                0
year              1205
manufacturer     17646
model             5277
condition       174104
cylinders       177678
fuel              3013
odometer          4400
title_status      8242
transmission      2556
VIN             161042
drive           130567
size            306361
type             92858
paint_color     130203
image_url           68
description         70
county          426880
state                0
lat               6549
long              6549
posting_date        68
dtype: int64

Nos damos cuenta que en la variable County, que está mal escrita, está llena de datos faltantes, y al no ser relevante en el estudio, es mejor eliminarla. También la variable de tamaño hay más del 70% de nulos y en VIN hay 37% de nulos.
Para el url de la imagen, descripción y fecha de publicación tienen muy pocos nulos, por lo que no implican un peligro en los siguientes pasos para nuestro EDA. Y las variables como id, url, y el url de la región se podrían categorizar como irrelevantes, por lo que se pueden eliminar.

Para variables numéricas como el año y el odómetro hay aprox. un 0.3% a 1% de datos faltantes, por lo que será mejor imputar con la mediana de estas mismas. Para la latitud  y la longitud se presenta un porcentaje de 1.5%, por lo que también se pueden imputar con la mediana.

Para las variables categóricas, como la manufactura, con un 4% de nulos, el modelo, con un 1.2% de nulos, el combustible, con un 0.7% de nulos, el título de estado, con un 2% de nulos, y la transmisión, con un 0.6% de nulos, se deben imputar con la moda. En las variables de condición, con un 41% de nulos se podría, el cilindraje, la conducción y el color de la puntura, o imputarlas con desconocido (unknown), o eliminarlas.

Por ello, al haber presencia de datos faltantes en la mayoría de variables, iniciamos su tratamiento para luego verificar distribución después de imputar para analizar si no se distorsionaron los datos.

In [7]:
print("Análisis de datos faltantes")
total_registros = len(dataset)
faltantes_por_columna = dataset.isnull().sum()
porcentaje_faltantes = (faltantes_por_columna / total_registros) * 100

analisis_faltantes = pd.DataFrame({
    'Valores_Faltantes': faltantes_por_columna,
    'Porcentaje_Faltantes': porcentaje_faltantes
})
print(analisis_faltantes.sort_values('Porcentaje_Faltantes', ascending=False))
print("\n")

variables_eliminar_faltantes = analisis_faltantes[analisis_faltantes['Porcentaje_Faltantes'] > 40].index.tolist()
variables_imputar = analisis_faltantes[(analisis_faltantes['Porcentaje_Faltantes'] > 0) & 
                                     (analisis_faltantes['Porcentaje_Faltantes'] <= 40)].index.tolist()
variables_limpias = analisis_faltantes[analisis_faltantes['Porcentaje_Faltantes'] == 0].index.tolist()

print("Clasificación de las variables")
print(f"Podemos eliminar la variable con un porcentaje aprox. mayor a 40% de faltantes: {len(variables_eliminar_faltantes)} variables")
for var in variables_eliminar_faltantes:
    print(f"   - {var}: {analisis_faltantes.loc[var, 'Porcentaje_Faltantes']:.1f}% faltantes")

print(f"Podemos imputar la variable con un porcentaje aprox. entre 0-40% de faltantes: {len(variables_imputar)} variables")
for var in variables_imputar:
    print(f"   - {var}: {analisis_faltantes.loc[var, 'Porcentaje_Faltantes']:.1f}% faltantes")

print(f"Variables limpias con un porcentaje de 0% de faltantes: {len(variables_limpias)} variables")
print("   -", ", ".join(variables_limpias))

Análisis de datos faltantes
              Valores_Faltantes  Porcentaje_Faltantes
county                   426880            100.000000
size                     306361             71.767476
cylinders                177678             41.622470
condition                174104             40.785232
VIN                      161042             37.725356
drive                    130567             30.586347
paint_color              130203             30.501078
type                      92858             21.752717
manufacturer              17646              4.133714
title_status               8242              1.930753
lat                        6549              1.534155
long                       6549              1.534155
model                      5277              1.236179
odometer                   4400              1.030735
fuel                       3013              0.705819
transmission               2556              0.598763
year                       1205              0.282281


Con las variables identificadas según su porcentaje de datos faltantes, empezamos con el tratamiento:

Primero, estudiamos las variables que para nuestro estudio pueden ser irrelevantes:

In [13]:
def identificar_variables_irrelevantes(dataset):
    variables_irrelevantes = [
        'id', 'url', 'region_url', 'image_url', 'description',
        'posting_date', 'VIN', 'county', 'lat', 'long'
    ]
    variables_irrelevantes = [col for col in variables_irrelevantes if col in dataset.columns]
    
    return variables_irrelevantes
variables_irrelevantes = identificar_variables_irrelevantes(dataset)
print("Variables irrelevantes:")
for i, col in enumerate(variables_irrelevantes, 1):
    print(f"{i}. {col}")

Variables irrelevantes:
1. id
2. url
3. region_url
4. image_url
5. description
6. posting_date
7. VIN
8. county
9. lat
10. long


Ahora procedemos a sus eliminaciones:

In [16]:
todas_variables_eliminar = list(set(variables_eliminar_faltantes + variables_irrelevantes))

print("Variables a eliminar:")
for i, var in enumerate(todas_variables_eliminar, 1):
    print(f"{i}. {var}")
print(f"\nDimensiones antes de eliminar: {dataset.shape}")
dataset_clean = dataset.drop(columns=todas_variables_eliminar, errors='ignore')
print(f"Dimensiones después de eliminar: {dataset_clean.shape}")

Variables a eliminar:
1. url
2. lat
3. image_url
4. posting_date
5. description
6. county
7. cylinders
8. condition
9. size
10. region_url
11. long
12. id
13. VIN

Dimensiones antes de eliminar: (426880, 26)
Dimensiones después de eliminar: (426880, 13)


También se encontró datos que son inválidos en el estudio, puede ser por un error de escritura.

In [19]:
print(f"Registros antes de filtrar: {len(dataset_clean)}")
precios_antes = len(dataset_clean)
dataset_clean = dataset_clean[(dataset_clean['price'] > 500) & (dataset_clean['price'] < 100000)]
precios_despues = len(dataset_clean)
print(f"Precios válidos: {precios_despues} registros ({precios_antes - precios_despues} eliminados)")

años_antes = len(dataset_clean)
dataset_clean = dataset_clean[(dataset_clean['year'] >= 1980) & (dataset_clean['year'] <= 2022)]
años_despues = len(dataset_clean)
print(f"Años válidos: {años_despues} registros ({años_antes - años_despues} eliminados)")

print(f"Registros después de filtrar: {len(dataset_clean)}")

Registros antes de filtrar: 373973
Precios válidos: 373973 registros (0 eliminados)
Años válidos: 373973 registros (0 eliminados)
Registros después de filtrar: 373973


Con estas variables eliminadas, revisamos de nuevo nuestra base:

In [20]:
faltantes_despues_limpieza = dataset_clean.isnull().sum()
porcentaje_faltantes_despues = (faltantes_despues_limpieza / len(dataset_clean)) * 100

analisis_despues = pd.DataFrame({
    'Valores_Faltantes': faltantes_despues_limpieza,
    'Porcentaje_Faltantes': porcentaje_faltantes_despues
})

variables_con_faltantes = analisis_despues[analisis_despues['Valores_Faltantes'] > 0]
print("Después de la limpieza:")
print(variables_con_faltantes.sort_values('Porcentaje_Faltantes', ascending=False))

Después de la limpieza:
              Valores_Faltantes  Porcentaje_Faltantes
drive                    112893             30.187473
paint_color              107465             28.736032
type                      78700             21.044300
manufacturer              12428              3.323235
title_status               6675              1.784888
model                      3623              0.968787
fuel                       2317              0.619563
odometer                   2032              0.543355
transmission               1744              0.466344


Vemos ahora un porcentaje que no afecta la continuidad de nuestro estudio. Ahora nos vamos a identificación de variables para imputar de acuerdo a su naturaleza

In [21]:
variables_a_imputar = variables_con_faltantes.index.tolist()

numeric_features = []
categorical_features = []

for col in variables_a_imputar:
    if dataset_clean[col].dtype in ['int64', 'float64']:
        numeric_features.append(col)
    else:
        categorical_features.append(col)

print("Variables numéricas que requieren imputar con mediana:")
for var in numeric_features:
    porcentaje = analisis_despues.loc[var, 'Porcentaje_Faltantes']
    print(f"• {var}: {porcentaje:.1f}% faltantes")

print("Variables categóricas que requieren imputar con moda:")
for var in categorical_features:
    porcentaje = analisis_despues.loc[var, 'Porcentaje_Faltantes']
    print(f"• {var}: {porcentaje:.1f}% faltantes")

Variables numéricas que requieren imputar con mediana:
• odometer: 0.5% faltantes
Variables categóricas que requieren imputar con moda:
• manufacturer: 3.3% faltantes
• model: 1.0% faltantes
• fuel: 0.6% faltantes
• title_status: 1.8% faltantes
• transmission: 0.5% faltantes
• drive: 30.2% faltantes
• type: 21.0% faltantes
• paint_color: 28.7% faltantes


Ahora las imputamos de acuerdo a lo que requieren

In [24]:
print("Imputación con mediana:")
if numeric_features:
    numeric_imputer = SimpleImputer(strategy='median')
    dataset_clean[numeric_features] = numeric_imputer.fit_transform(dataset_clean[numeric_features])
    print(f"Imputadas: {', '.join(numeric_features)}")

print("Imputación con moda:")
if categorical_features:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    dataset_clean[categorical_features] = categorical_imputer.fit_transform(dataset_clean[categorical_features])
    print(f"Imputadas: {', '.join(categorical_features)}")

# Verificar que no queden valores faltantes
faltantes_finales = dataset_clean.isnull().sum().sum()
print(f"Valores faltantes restantes: {faltantes_finales}")

Imputación con mediana:
Imputadas: odometer
Imputación con moda:
Imputadas: manufacturer, model, fuel, title_status, transmission, drive, type, paint_color
Valores faltantes restantes: 0


Vemos cómo quedan al final:

In [26]:
dataset_clean['HighDemand'] = (dataset_clean['price'] > dataset_clean['price'].median()).astype(int)
print(f"Dimensiones finales: {dataset_clean.shape}")
print(f"Rango de precios: ${dataset_clean['price'].min():,} - ${dataset_clean['price'].max():,}")
print(f"Años: {int(dataset_clean['year'].min())} - {int(dataset_clean['year'].max())}")
print(f"Distribución de HighDemand:")
print(dataset_clean['HighDemand'].value_counts())
print(f"Porcentaje de alta demanda: {dataset_clean['HighDemand'].mean()*100:.1f}%")

Dimensiones finales: (373973, 14)
Rango de precios: $501 - $99,999
Años: 1980 - 2022
Distribución de HighDemand:
HighDemand
0    187140
1    186833
Name: count, dtype: int64
Porcentaje de alta demanda: 50.0%
